# 1. Установка системных зависимостей
Скачиваем системные пакеты, устанавливаем Ollama и нужные библиотеки Python.

In [10]:
print("Установка системных зависимостей и Ollama...")
!apt-get update && apt-get install -y zstd lshw pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama gradio faiss-cpu pypdf numpy
print("Установка завершена!")

Установка системных зависимостей и Ollama...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (1,874 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu j

# 2. Импорт библиотек и подключение Google Диска
Загружаем установленные модули и даем доступ к файлам на Google Drive.

In [11]:
import os
import subprocess
import time
import numpy as np
import faiss
import ollama
import gradio as gr
from pypdf import PdfReader
from google.colab import drive

# Подключаем гугл-диск
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3. Подготовка GPU и запуск сервера Ollama
Настраиваем видеокарту и запускаем локальный сервер нейросетей в фоновом режиме.

In [12]:
print("Подготовка окружения GPU...")
!pkill -9 ollama

os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia"
os.environ["PATH"] += ":/usr/local/cuda/bin"
os.environ["OLLAMA_INTEL_GPU"] = "0"

print("Запуск сервера Ollama...")
process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE)

# Даем серверу 15 секунд на запуск
time.sleep(15)
print("Сервер готов к работе.")

Подготовка окружения GPU...
Запуск сервера Ollama...
Сервер готов к работе.


# 4. Загрузка нейросетей
Скачиваем модель для поиска (bge-m3) и модель для генерации ответов (qwen3:4b).

In [13]:
print("Загрузка моделей (это может занять время)...")
!ollama pull bge-m3
!ollama pull qwen3:4b

print("\n--- Проверка состояния видеокарты ---")
!nvidia-smi

print("\n--- Список загруженных моделей ---")
!ollama list

Загрузка моделей (это может занять время)...



--- Проверка состояния видеокарты ---
Mon Apr 27 13:48:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |           

# 5. Загрузка векторной базы данных 3GPP
Указываем пути к файлам на диске и загружаем индекс FAISS вместе с текстами.

In [14]:
PDF_DIR = "/content/drive/MyDrive/RAG_System/Database"
DB_DIR = "/content/drive/MyDrive/RAG_System/Vector_embeddings_database/3GPP_Database"
MODEL_NAME = "qwen3:4b"
EMBED_MODEL = "bge-m3"

client = ollama.Client(host='http://127.0.0.1:11434')

def load_db():
    index_path = os.path.join(DB_DIR, "index.faiss")
    texts_path = os.path.join(DB_DIR, "texts.npy")
    if os.path.exists(index_path) and os.path.exists(texts_path):
        index = faiss.read_index(index_path)
        chunks = np.load(texts_path, allow_pickle=True, mmap_mode='r')
        print("База данных успешно загружена!")
        return index, chunks

    print("Ошибка: База не найдена по указанному пути.")
    return None, None

index, all_chunks = load_db()

Ошибка: База не найдена по указанному пути.


# 6. Основная логика RAG
Здесь описаны функции для поиска информации по базе и генерации ответа от лица инженера 5G.

In [15]:
def get_context(query, index, chunks, k=3):
    resp = client.embeddings(model=EMBED_MODEL, prompt=query)
    query_emb = np.array([resp['embedding']]).astype('float32')
    _, indices = index.search(query_emb, k)
    context_parts = [str(chunks[i]) for i in indices[0] if i != -1 and i < len(chunks)]
    return "\n---\n".join(context_parts)

def chat(message, history):
    try:
        context = get_context(message, index, all_chunks, k=4)

        raw_prompt = (
            f"<|im_start|>system\n"
            f"Ты — ведущий инженер по сетям 5G и эксперт по стандартам 3GPP. "
            f"Твоя цель: дать развернутый, понятный и технически грамотный ответ на русском языке. \n\n"
            f"ПРАВИЛА:\n"
            f"1. Используй предоставленную ДОКУМЕНТАЦИЮ как главный источник истины.\n"
            f"2. Пиши структурированно: используй заголовки, жирный шрифт для терминов и маркированные списки.\n"
            f"3. Если в тексте есть сложные аббревиатуры (например, AMF, gNB, PDU), кратко поясняй их значение.\n"
            f"4. Не используй технические теги в ответе. Твой тон: профессиональный, дружелюбный, экспертный.\n"
            f"5. Если информации недостаточно, ответь на основе того, что есть.\n"
            f"<|im_end|>\n"
            f"<|im_start|>user\n"
            f"КОНТЕКСТ ИЗ ДОКУМЕНТАЦИИ:\n{context}\n\n"
            f"ВОПРОС ПОЛЬЗОВАТЕЛЯ: {message}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )

        stream = client.generate(
            model=MODEL_NAME,
            prompt=raw_prompt,
            stream=True,
            options={
                "num_ctx": 4096,
                "temperature": 0.4,
                "top_p": 0.9,
                "stop": ["<|im_start|>", "<|im_end|>", "system\n", "user\n"]
            }
        )

        full_response = ""
        for chunk in stream:
            content = chunk.get('response', '')
            full_response += content
            yield full_response.strip()

    except Exception as e:
        yield f"Произошла техническая ошибка: {str(e)}"

# 7. Запуск интерфейса Gradio
Создаем веб-интерфейс чат-бота и выводим публичную ссылку.

In [ ]:
demo = gr.ChatInterface(
    fn=chat,
    title="📡 5G Ассистент (RAG система)",
    description="Локальный доступ к базе 3GPP через языковую модель Qwen3:4B.",
    type="messages" # Избавит от предупреждений Gradio в логах
)

if __name__ == "__main__":
    gr.close_all()
    print("Запускаем веб-интерфейс...")
    demo.launch(
        share=False,
        server_name="127.0.0.1",
        server_port=7860,
        debug=True
    )

Closing server running on port: 7860
Запускаем веб-интерфейс...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>